# Building a Local Lakehouse with Delta Lake

**BINFX410 — Chapter 10, Notebook 4 of 5**

---

In this notebook we build a **lakehouse** — the modern unified architecture that combines the low-cost open storage of a data lake with the reliability guarantees of a data warehouse. We use **Delta Lake**, the most widely adopted open-source lakehouse table format, via the Python `deltalake` library.

Everything runs **100% locally** — no Spark, no cloud, no cluster required.

**Prerequisites:** Run `01_introduction_and_setup.ipynb` to generate `./raw_data/`.

## Learning Objectives

1. Explain the lakehouse architecture and why it was created.
2. Understand the Delta Lake transaction log (`_delta_log`).
3. Write and read Delta tables using the Python `deltalake` library.
4. Demonstrate ACID transactions, schema enforcement, and time travel.
5. Perform upsert/merge operations on Delta tables.
6. Query Delta tables with DuckDB.
7. Run OPTIMIZE and VACUUM operations.

## 1. Lakehouse Concepts

### 1.1 The Core Problem: Why Not Just a Data Lake?

Data lakes on object storage (S3, ADLS, GCS) are cheap and flexible, but they have fundamental limitations:

```
Data Lake Problem                      Delta Lake Solution
─────────────────                      ──────────────────
No ACID — partial writes visible   →   Transaction log ensures atomicity
No schema enforcement              →   Schema validation on every write
Can't update/delete rows           →   MERGE (upsert), UPDATE, DELETE
No history — overwrite = data loss →   Time travel: query any past version
Small file problem (Spark)         →   OPTIMIZE compacts small files
Stale data accumulates             →   VACUUM removes old file versions
```

### 1.2 The Delta Lake Transaction Log

The key innovation in Delta Lake is the **transaction log** (`_delta_log/`). Every operation that changes the table is recorded as a JSON file:

```
samples/
├── _delta_log/
│   ├── 00000000000000000000.json   ← initial write (v0)
│   ├── 00000000000000000001.json   ← append new samples (v1)
│   ├── 00000000000000000002.json   ← schema change (v2)
│   ├── 00000000000000000003.json   ← MERGE update (v3)
│   └── 00000000000000000010.checkpoint.parquet  ← checkpoint every 10
├── part-00000-abc123.parquet       ← actual data files
├── part-00001-def456.parquet
└── part-00002-ghi789.parquet
```

Each log entry records:
- Which Parquet files were **added**
- Which Parquet files were **removed** (logically deleted)
- Metadata changes (schema, table properties)
- Commit timestamp and operation type

**Reading version N**: replay the log from 0 to N to determine which files are "live" at that version.

### 1.3 ACID Properties in Delta Lake

| Property | Meaning | Delta Lake mechanism |
|----------|---------|-----------------------|
| **Atomicity** | All or nothing — no partial writes | Log entry committed only after all files written |
| **Consistency** | Data always valid | Schema enforcement on write |
| **Isolation** | Concurrent readers see consistent snapshot | Readers use a specific log version |
| **Durability** | Committed data persists | Parquet files + log are on durable storage |

## 2. Setup

In [7]:
import os
import json
import shutil
import pandas as pd
import pyarrow as pa
import duckdb

from deltalake import DeltaTable, write_deltalake  # type: ignore

# Clean slate for the lakehouse directory
LAKEHOUSE_ROOT = './lakehouse'
if os.path.exists(LAKEHOUSE_ROOT):
    shutil.rmtree(LAKEHOUSE_ROOT)
os.makedirs(LAKEHOUSE_ROOT)

import deltalake
print(f'deltalake version : {deltalake.__version__}')
print(f'pyarrow version   : {pa.__version__}')
print(f'Lakehouse root    : {LAKEHOUSE_ROOT}')

deltalake version : 1.5.0
pyarrow version   : 21.0.0
Lakehouse root    : ./lakehouse


## 3. Writing Delta Tables

We write each genomics dataset as a Delta table. `write_deltalake()` accepts a pandas DataFrame or PyArrow Table and creates the Parquet files plus the `_delta_log/`.

In [8]:
# Load source data
df_samples = pd.read_csv('./raw_data/samples.csv')
df_genes   = pd.read_csv('./raw_data/genes.csv')
df_calls   = pd.read_csv('./raw_data/variant_calls.csv')
df_variants = pd.read_csv('./raw_data/variants.csv')

# Cast date columns to proper types
df_samples['collection_date'] = pd.to_datetime(df_samples['collection_date'])
df_calls['call_date']         = pd.to_datetime(df_calls['call_date'])

print('Source data loaded.')
for name, df in [('samples', df_samples), ('genes', df_genes),
                 ('variant_calls', df_calls), ('variants', df_variants)]:
    print(f'  {name:<18}: {len(df):,} rows')

Source data loaded.
  samples           : 500 rows
  genes             : 100 rows
  variant_calls     : 2,000 rows
  variants          : 5,000 rows


In [9]:
# Write samples Delta table
SAMPLES_PATH = f'{LAKEHOUSE_ROOT}/samples'

write_deltalake(
    SAMPLES_PATH,
    df_samples,
    mode='overwrite',
)

print(f'Samples Delta table written to: {SAMPLES_PATH}')
print('\nFiles created:')
for root, dirs, files in os.walk(SAMPLES_PATH):
    level = root.replace(SAMPLES_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  {f:<50}  ({size:,} bytes)')

Samples Delta table written to: ./lakehouse/samples

Files created:
samples/
  part-00000-3173cf0a-6f00-427e-a112-593452dd805f-c000.snappy.parquet  (17,978 bytes)
  _delta_log/
    00000000000000000000.json                           (2,439 bytes)


In [10]:
# Write remaining Delta tables
for name, df, path in [
    ('genes',         df_genes,    f'{LAKEHOUSE_ROOT}/genes'),
    ('variant_calls', df_calls,    f'{LAKEHOUSE_ROOT}/variant_calls'),
    ('variants',      df_variants, f'{LAKEHOUSE_ROOT}/variants'),
]:
    write_deltalake(path, df, mode='overwrite')
    dt = DeltaTable(path)
    print(f'{name:<18}: version={dt.version()}, files={dt.get_add_actions().num_rows}')

genes             : version=0, files=1
variant_calls     : version=0, files=1
variants          : version=0, files=1


## 4. Examining the Transaction Log

Let's read the `_delta_log` JSON files manually to understand what Delta Lake records.

In [11]:
log_path = f'{SAMPLES_PATH}/_delta_log/00000000000000000000.json'

with open(log_path) as f:
    log_lines = f.readlines()

print(f'Transaction log v0 — {len(log_lines)} entries\n')

for i, line in enumerate(log_lines[:6]):  # show first 6 entries
    entry = json.loads(line)
    action = list(entry.keys())[0]
    print(f'--- Entry {i+1}: action = "{action}" ---')
    print(json.dumps(entry, indent=2)[:600])  # truncate for readability
    print()

Transaction log v0 — 4 entries

--- Entry 1: action = "commitInfo" ---
{
  "commitInfo": {
    "timestamp": 1774724051137,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "Overwrite"
    },
    "engineInfo": "delta-rs:py-1.5.0",
    "clientVersion": "delta-rs.py-1.5.0",
    "operationMetrics": {
      "execution_time_ms": 1,
      "num_added_files": 1,
      "num_added_rows": 500,
      "num_partitions": 0,
      "num_removed_files": 0
    }
  }
}

--- Entry 2: action = "protocol" ---
{
  "protocol": {
    "minReaderVersion": 3,
    "minWriterVersion": 7,
    "readerFeatures": [
      "timestampNtz"
    ],
    "writerFeatures": [
      "timestampNtz"
    ]
  }
}

--- Entry 3: action = "metaData" ---
{
  "metaData": {
    "id": "ed21f40c-a04b-4c82-a5ad-782e185d274f",
    "name": null,
    "description": null,
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"sample_id\",\"type\":

In [12]:
# Inspect the table using the DeltaTable API
dt_samples = DeltaTable(SAMPLES_PATH)

print('=== DeltaTable Metadata ===')
print(f'Current version : {dt_samples.version()}')
print(f'Active files    : {dt_samples.get_add_actions().num_rows}')
print(f'\nSchema:')
print(dt_samples.schema())
print(f'\nTable metadata:')
meta = dt_samples.metadata()
print(f'  id          : {meta.id}')
print(f'  name        : {meta.name}')
print(f'  created at  : {meta.created_time}')

=== DeltaTable Metadata ===
Current version : 0
Active files    : 1

Schema:
Schema([Field(sample_id, PrimitiveType("long"), nullable=True), Field(patient_id, PrimitiveType("long"), nullable=True), Field(tissue_type, PrimitiveType("string"), nullable=True), Field(sequencing_platform, PrimitiveType("string"), nullable=True), Field(library_prep, PrimitiveType("string"), nullable=True), Field(collection_date, PrimitiveType("timestamp_ntz"), nullable=True), Field(mean_coverage, PrimitiveType("double"), nullable=True), Field(pct_bases_20x, PrimitiveType("double"), nullable=True), Field(project_id, PrimitiveType("string"), nullable=True)])

Table metadata:
  id          : ed21f40c-a04b-4c82-a5ad-782e185d274f
  name        : None
  created at  : 1774724051136


## 5. ACID Transactions — Demonstrating Atomicity

In [13]:
# Demonstrate append — adds new samples as a new transaction
# Simulate 3 new samples collected in 2025

new_samples = pd.DataFrame([
    {'sample_id': 501, 'patient_id': 301, 'tissue_type': 'Tumor',
     'sequencing_platform': 'Illumina_NovaSeq', 'library_prep': 'WGS',
     'collection_date': pd.Timestamp('2025-01-15'),
     'mean_coverage': 95.3, 'pct_bases_20x': 0.9421, 'project_id': 'PROJ_001'},
    {'sample_id': 502, 'patient_id': 302, 'tissue_type': 'Blood',
     'sequencing_platform': 'Illumina_HiSeq', 'library_prep': 'WES',
     'collection_date': pd.Timestamp('2025-02-03'),
     'mean_coverage': 58.1, 'pct_bases_20x': 0.8834, 'project_id': 'PROJ_003'},
    {'sample_id': 503, 'patient_id': 303, 'tissue_type': 'Normal',
     'sequencing_platform': 'Oxford_Nanopore', 'library_prep': 'WGS',
     'collection_date': pd.Timestamp('2025-02-20'),
     'mean_coverage': 42.7, 'pct_bases_20x': 0.8190, 'project_id': 'PROJ_005'},
])

# Append to the existing Delta table (does NOT overwrite)
write_deltalake(SAMPLES_PATH, new_samples, mode='append')

dt_samples = DeltaTable(SAMPLES_PATH)
print(f'After append:')
print(f'  Version : {dt_samples.version()}')
print(f'  Files   : {dt_samples.get_add_actions().num_rows}')

df_full = dt_samples.to_pandas()
print(f'  Total rows: {len(df_full):,}')
print()
print('New samples appended:')
print(df_full[df_full['sample_id'] >= 501])

After append:
  Version : 1
  Files   : 2
  Total rows: 503

New samples appended:
   sample_id  patient_id tissue_type sequencing_platform library_prep  \
0        501         301       Tumor    Illumina_NovaSeq          WGS   
1        502         302       Blood      Illumina_HiSeq          WES   
2        503         303      Normal     Oxford_Nanopore          WGS   

  collection_date  mean_coverage  pct_bases_20x project_id  
0      2025-01-15           95.3         0.9421   PROJ_001  
1      2025-02-03           58.1         0.8834   PROJ_003  
2      2025-02-20           42.7         0.8190   PROJ_005  


## 6. Schema Enforcement

Delta Lake **enforces the schema** of the table. If you try to write data with wrong column types or missing required columns, it raises an error rather than silently corrupting data. This is in stark contrast to a plain data lake.

In [14]:
# Attempt 1: Write data with an extra column NOT in the schema
bad_sample_extra_col = pd.DataFrame([{
    'sample_id': 999,
    'patient_id': 999,
    'tissue_type': 'Tumor',
    'sequencing_platform': 'Illumina_NovaSeq',
    'library_prep': 'WGS',
    'collection_date': pd.Timestamp('2025-03-01'),
    'mean_coverage': 80.0,
    'pct_bases_20x': 0.92,
    'project_id': 'PROJ_001',
    'sequencing_lab': 'LabX',  # extra column — not in schema!
}])

print('Attempting to write data with extra column (no schema_mode specified)...')
try:
    write_deltalake(SAMPLES_PATH, bad_sample_extra_col, mode='append')
    print('  Write succeeded (schema was merged — use schema_mode=\'merge\' explicitly for this)')
except Exception as e:
    print(f'  Write BLOCKED: {type(e).__name__}: {e}')

Attempting to write data with extra column (no schema_mode specified)...
  Write BLOCKED: SchemaMismatchError: Cannot cast schema, number of fields does not match: 10 vs 9


In [15]:
# Attempt 2: Write data with wrong type for sample_id (string instead of int)
bad_type = pd.DataFrame([{
    'sample_id': 'NOT_AN_INT',  # wrong type!
    'patient_id': 1,
    'tissue_type': 'Tumor',
    'sequencing_platform': 'Illumina_NovaSeq',
    'library_prep': 'WGS',
    'collection_date': pd.Timestamp('2025-03-01'),
    'mean_coverage': 80.0,
    'pct_bases_20x': 0.92,
    'project_id': 'PROJ_001',
}])

print('Attempting to write data with wrong type for sample_id...')
try:
    write_deltalake(SAMPLES_PATH, bad_type, mode='append')
    print('  Write succeeded')
except Exception as e:
    print(f'  Write BLOCKED: {type(e).__name__}')
    print(f'  Message: {str(e)[:300]}')

print()
print(f'Table still at version: {DeltaTable(SAMPLES_PATH).version()} (unchanged)')

Attempting to write data with wrong type for sample_id...
  Write BLOCKED: Exception
  Message: Cast error: Cannot cast string 'NOT_AN_INT' to value of Int64 type

Table still at version: 1 (unchanged)


## 7. Schema Evolution

Unlike schema enforcement (which blocks bad writes), **schema evolution** allows intentional, controlled changes to the schema using `schema_mode='merge'`. This is the correct way to add new columns.

In [ ]:
# Intentional schema evolution: new batch of genes arrives with 'transcript_count' column
GENES_PATH = f'{LAKEHOUSE_ROOT}/genes'

df_genes_v2 = pd.DataFrame([{
    'gene_id':        101,
    'gene_name':      'NOVEL1',
    'chromosome':     'chr3',
    'start_pos':      12345678,
    'end_pos':        12445678,
    'strand':         '+',
    'biotype':        'protein_coding',
    'is_cancer_gene': False,
    'transcript_count': 4,  # new column from upgraded annotation pipeline!
}])

print('Schema before evolution:')
print(DeltaTable(GENES_PATH).schema())
print()

# schema_mode='merge' explicitly allows adding new columns
write_deltalake(GENES_PATH, df_genes_v2, mode='append', schema_mode='merge')

print('Schema after evolution (transcript_count column added):')
dt_genes = DeltaTable(GENES_PATH)
print(dt_genes.schema())
print(f'\nVersion: {dt_genes.version()}')

# Old rows will have NULL for the new 'transcript_count' column
df_all_genes = dt_genes.to_pandas()
print(f'\nRows with transcript_count value : {df_all_genes["transcript_count"].notna().sum()}')
print(f'Rows with NULL transcript_count  : {df_all_genes["transcript_count"].isna().sum()}')

## 8. Time Travel — Querying Historical Versions

Time travel is one of the most powerful Delta Lake features. Because the transaction log records every change, you can query any previous version of the table — perfect for auditing, debugging, and reproducibility.

In [ ]:
# Show the full version history of the samples table
dt_samples = DeltaTable(SAMPLES_PATH)

print('=== Samples Table Version History ===')
history = dt_samples.history()
for entry in history:
    ts        = entry.get('timestamp', 'N/A')
    operation = entry.get('operation', 'N/A')
    version   = entry.get('version', 'N/A')
    print(f'  v{version}  {operation:<12}  {ts}')

In [ ]:
# Read version 0 (original write — 500 samples)
dt_v0 = DeltaTable(SAMPLES_PATH, version=0)
df_v0 = dt_v0.to_pandas()

# Read current version
dt_current = DeltaTable(SAMPLES_PATH)
df_current = dt_current.to_pandas()

print(f'Version 0 row count   : {len(df_v0):,}')
print(f'Current version       : {dt_current.version()}')
print(f'Current row count     : {len(df_current):,}')
print(f'Rows added since v0   : {len(df_current) - len(df_v0)}')
print()

# The new samples (501, 502, 503) don't exist in version 0
new_ids = [501, 502, 503]
in_v0      = df_v0[df_v0['sample_id'].isin(new_ids)]
in_current = df_current[df_current['sample_id'].isin(new_ids)]

print(f'New samples in v0      : {len(in_v0)} (should be 0)')
print(f'New samples in current : {len(in_current)} (should be 3)')
print()
print('New samples visible in current version:')
print(in_current[['sample_id', 'patient_id', 'tissue_type', 'project_id']])

## 9. Upsert / MERGE Operations

MERGE is the operation that enables true upsert (update-or-insert) logic. In a data lake, you can't update a row — you'd have to rewrite the entire file. Delta Lake handles this efficiently by writing a new file and marking old files as removed in the transaction log.

In [ ]:
# Scenario: quality re-evaluation has changed filter status for some variant_calls
# Some 'Artifact' calls have been re-reviewed and confirmed as 'PASS'
# Two new variant calls are also being added

CALLS_PATH = f'{LAKEHOUSE_ROOT}/variant_calls'
dt_calls = DeltaTable(CALLS_PATH)
df_calls_now = dt_calls.to_pandas()

# Find some Artifact calls to update
artifact_calls = df_calls_now[df_calls_now['filter_status'] == 'Artifact'].head(2)
print('Artifact calls to re-evaluate as PASS:')
print(artifact_calls[['call_id', 'sample_id', 'caller', 'filter_status']].to_string(index=False))

In [ ]:
# Build the updates DataFrame: change Artifact → PASS for reviewed calls
updates = artifact_calls.copy()
updates['filter_status'] = 'PASS'

# Add 2 new variant calls
new_calls = pd.DataFrame([
    {'call_id': 2001, 'sample_id': 1,   'call_date': pd.Timestamp('2025-03-01'),
     'caller': 'GATK',        'filter_status': 'PASS'},
    {'call_id': 2002, 'sample_id': 2,   'call_date': pd.Timestamp('2025-03-05'),
     'caller': 'Mutect2',     'filter_status': 'PASS'},
])

all_updates = pd.concat([updates, new_calls], ignore_index=True)
print('All updates to merge:')
print(all_updates)

In [ ]:
# Perform the MERGE operation
dt_calls = DeltaTable(CALLS_PATH)
updates_arrow = pa.Table.from_pandas(all_updates)

(
    dt_calls.merge(
        source=updates_arrow,
        predicate='target.call_id = source.call_id',
        source_alias='source',
        target_alias='target',
    )
    .when_matched_update_all()      # update all columns when call_ids match
    .when_not_matched_insert_all()  # insert when no match (new calls)
    .execute()
)

dt_calls = DeltaTable(CALLS_PATH)
print(f'After MERGE:')
print(f'  Version   : {dt_calls.version()}')
print(f'  Total rows: {len(dt_calls.to_pandas()):,}')

df_merged = dt_calls.to_pandas()

# Verify updates applied
updated_ids = artifact_calls['call_id'].tolist()
print('\nUpdated calls (should show PASS):')
print(df_merged[df_merged['call_id'].isin(updated_ids)][['call_id', 'caller', 'filter_status']])

print('\nInserted new calls (2001, 2002):')
print(df_merged[df_merged['call_id'].isin([2001, 2002])][['call_id', 'sample_id', 'caller', 'filter_status']])

## 10. Querying Delta Tables with DuckDB

DuckDB can query Delta tables directly using the `delta_scan()` function. This gives you full SQL analytics power on top of Delta Lake.

In [ ]:
# Install and load the delta extension
con = duckdb.connect()  # in-memory
con.execute('INSTALL delta; LOAD delta;')

print('DuckDB delta extension loaded.')

# Query 1: Variant counts by biotype (PASS calls only)
result = con.execute("""
    SELECT
        g.biotype,
        COUNT(v.variant_id)          AS variant_count,
        ROUND(AVG(v.af_tumor), 4)    AS mean_af_tumor,
        COUNT(DISTINCT v.call_id)    AS unique_calls
    FROM delta_scan('./lakehouse/variants')      v
    JOIN delta_scan('./lakehouse/genes')         g  ON v.gene_id  = g.gene_id
    JOIN delta_scan('./lakehouse/variant_calls') c  ON v.call_id  = c.call_id
    WHERE c.filter_status = 'PASS'
    GROUP BY g.biotype
    ORDER BY variant_count DESC
""").df()

print('=== Variant Counts by Biotype (PASS only, via DuckDB on Delta) ===')
print(result)

In [ ]:
# Query 2: Top 5 samples by total PASS variant count
top_samples = con.execute("""
    SELECT
        s.sample_id,
        s.tissue_type,
        s.project_id,
        s.mean_coverage,
        COUNT(v.variant_id)          AS total_variants,
        ROUND(AVG(v.af_tumor), 4)    AS mean_af_tumor
    FROM delta_scan('./lakehouse/samples')       s
    JOIN delta_scan('./lakehouse/variant_calls') c  ON s.sample_id = c.sample_id
    JOIN delta_scan('./lakehouse/variants')      v  ON c.call_id   = v.call_id
    WHERE c.filter_status = 'PASS'
    GROUP BY s.sample_id, s.tissue_type, s.project_id, s.mean_coverage
    ORDER BY total_variants DESC
    LIMIT 5
""").df()

print('=== Top 5 Samples by PASS Variant Count ===')
print(top_samples)

## 11. OPTIMIZE and VACUUM

Over time, Delta tables accumulate many small Parquet files (especially from streaming ingestion or frequent small appends). `OPTIMIZE` compacts these into larger files for better query performance. `VACUUM` physically deletes old data files that are no longer referenced by any version within the retention window.

In [ ]:
# Show current file count before optimize
dt_samples = DeltaTable(SAMPLES_PATH)
active_files = dt_samples.get_add_actions().column('path').to_pylist()
print(f'Files before OPTIMIZE: {len(active_files)}')
for f in active_files:
    fpath = f'{SAMPLES_PATH}/{f}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        print(f'  {f[-50:]:50s}  {size:,} bytes')

In [ ]:
# OPTIMIZE: compact small files into fewer larger files
optimize_result = dt_samples.optimize.compact()

print('OPTIMIZE result:')
print(f'  metrics: {optimize_result}')

dt_samples = DeltaTable(SAMPLES_PATH)
print(f'\nFiles after OPTIMIZE: {dt_samples.get_add_actions().num_rows}')

In [ ]:
# VACUUM: remove files no longer referenced by any version
# retention_hours=0 forces deletion of all unreferenced files immediately
# In production, keep at least 168 hours (7 days) to protect time travel

all_files_before = []
for root, dirs, files in os.walk(SAMPLES_PATH):
    for f in files:
        if f.endswith('.parquet'):
            all_files_before.append(os.path.join(root, f))

print(f'Parquet files on disk (before VACUUM): {len(all_files_before)}')

vacuum_result = dt_samples.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=False)

all_files_after = []
for root, dirs, files in os.walk(SAMPLES_PATH):
    for f in files:
        if f.endswith('.parquet'):
            all_files_after.append(os.path.join(root, f))

print(f'Parquet files on disk (after VACUUM) : {len(all_files_after)}')
print(f'Files deleted                        : {len(all_files_before) - len(all_files_after)}')
print()
print('WARNING: After VACUUM with short retention, time travel to old versions')
print('is no longer possible. In production, keep retention >= 7 days.')

## 12. Lakehouse Summary

In [ ]:
print('=== Lakehouse Summary ===')
print(f'Root: {LAKEHOUSE_ROOT}/')
print()

for table_name in ['samples', 'genes', 'variant_calls', 'variants']:
    path = f'{LAKEHOUSE_ROOT}/{table_name}'
    dt   = DeltaTable(path)
    df   = dt.to_pandas()
    log_files = os.listdir(f'{path}/_delta_log')
    print(f'  {table_name:<18}  v{dt.version()}  {len(df):>6,} rows  {dt.get_add_actions().num_rows} data files  {len(log_files)} log entries')

print()
print('Key capabilities demonstrated:')
print('  ACID transactions     - transaction log ensures atomicity')
print('  Schema enforcement    - bad writes blocked')
print('  Schema evolution      - new columns added safely with merge mode')
print('  Time travel           - query any historical version')
print('  MERGE (upsert)        - update existing + insert new in one operation')
print('  DuckDB integration    - SQL analytics directly on Delta tables')
print('  OPTIMIZE + VACUUM     - file compaction and cleanup')

## Exercises

**Exercise 4.1 — Time Travel Audit**

The samples table has been through several versions (initial write, append of 3 new samples).

a) Use `DeltaTable(SAMPLES_PATH, version=N).to_pandas()` to count rows at each version (0, 1, 2, ..., current).

b) Find samples whose `tissue_type` changed between version 0 and the current version. Hint: merge both DataFrames on `sample_id` and compare the `tissue_type` column.

In [ ]:
# Exercise 4.1 — Time travel audit
# a) Row counts at each version
# dt_samples = DeltaTable(SAMPLES_PATH)
# current_version = dt_samples.version()
# for v in range(current_version + 1):
#     ...

# b) Changed records between v0 and current
# df_v0 = DeltaTable(SAMPLES_PATH, version=0).to_pandas()
# df_now = DeltaTable(SAMPLES_PATH).to_pandas()
# ...

**Exercise 4.2 — Bulk MERGE on Variant Calls**

Write a MERGE operation on the `variant_calls` table that:
- Updates the `filter_status` from `'LowQuality'` to `'Reanalysis_Needed'` for all calls where `call_date` is before 2022-01-01 (these old calls need to be re-run with the new pipeline)
- Does NOT insert new rows (only update existing)

Hint: Create a DataFrame of affected calls with the updated filter_status, then use `.when_matched_update_all()`.

In [ ]:
# Exercise 4.2 — Bulk MERGE on variant_calls
CALLS_PATH = f'{LAKEHOUSE_ROOT}/variant_calls'
dt_calls_ex = DeltaTable(CALLS_PATH)

# Step 1: Find LowQuality calls before 2022
# df_calls_now = dt_calls_ex.to_pandas()
# cutoff = pd.Timestamp('2022-01-01')
# to_reanalyze = ...

# Step 2: Perform the merge (update only, no inserts)
# (dt_calls_ex.merge(...)...execute())
print('TODO: implement Exercise 4.2')

**Exercise 4.3 — Cohort Retention via Delta Tables**

Using DuckDB's `delta_scan()`, write a query that produces a **monthly cohort retention table** for samples: for each collection cohort (year+month), what percentage of samples had variant calls in month 0, 1, 2, and 3 after collection?

Expected output columns: `project_id`, `cohort_year`, `cohort_month`, `month_0_pct`, `month_1_pct`, `month_2_pct`, `month_3_pct`.

In [ ]:
# Exercise 4.3 — Cohort retention via DuckDB on Delta tables
con = duckdb.connect()
con.execute('INSTALL delta; LOAD delta;')

retention = con.execute("""
    -- TODO: write cohort retention query
    SELECT 'replace me' AS placeholder
""").df()
print(retention)

---

**Next:** `05_comparison_and_capstone.ipynb` — Performance benchmarks and capstone project